<a href="https://colab.research.google.com/github/hmsquare/DHmethods2/blob/main/261014_dh2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

研究テーマ：独学者向けの機械翻訳システムの開発

構想：WikipediaなどのWebページを機械翻訳で独学者のレベルに合わせて学習用の文章を生成する

（例：英語からやさしい日本語に翻訳する、が、「やさしい」の定義が未定）

現時点の考え：Wikipediaからデータを取り、原文の難しさが翻訳結果にどのような影響を与えているのかを検証


# Setup

Initialize output location

In [ ]:
from pathlib import Path

OUT = Path("_out")
OUT.mkdir(exist_ok=True)

Check /content of session

In [ ]:
!ls -lah /content

total 20K
drwxr-xr-x 1 root root 4.0K Oct 28 08:02 .
drwxr-xr-x 1 root root 4.0K Oct 28 07:42 ..
drwxr-xr-x 4 root root 4.0K Oct 26 22:18 .config
drwxr-xr-x 2 root root 4.0K Oct 28 08:02 _out
drwxr-xr-x 1 root root 4.0K Oct 26 22:18 sample_data


# 261014 データ収集
Webページの取得・データ整形を、授業で扱ったものをアレンジするか、自身の研究テーマに置き換えて実行した結果を共有してください。

In [ ]:
import requests
from bs4 import BeautifulSoup

# Target URL to scrape
url = "https://en.wikipedia.org/wiki/Wikipedia"

try:
    # Send a GET request to download the HTML
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status() # Check for request errors

    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # Print the page title
    print("=== Page Title ===")
    print(soup.title.string if soup.title else "No title found")
    print("\n" + "="*40 + "\n")

    # Print the first 1000 characters of the formatted HTML structure
    print("=== Pretty-Printed HTML (Truncated) ===")
    print(soup.prettify()[:1000] + "\n... [truncated] ...")

except Exception as e:
    print(f"An error occurred while fetching the URL: {e}")

=== Page Title ===
Wikipedia - Wikipedia


=== Pretty-Printed HTML (Truncated) ===
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Wikipedia - Wikipedia
  </title>
  <script>
   (function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disab

In [ ]:
# Save the raw HTML content to a file in the output directory
html_file_path = OUT / "wiki.html"

try:
    # We write response.text (the raw HTML) to the file
    html_file_path.write_text(response.text, encoding="utf-8")
    print(f"Successfully saved HTML to: {html_file_path}")
except NameError:
    print("Make sure you have run the scraping cell first to define the 'response' variable.")

Successfully saved HTML to: _out/wiki.html


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

# Load the saved HTML file and parse it
with open(html_file_path, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

data = []

# Find all heading tags in the page to map structure
h2_elements = soup.find_all("h2")
h2s = [h2.get_text() for h2 in h2_elements]
h2s

['Contents',
 'History',
 'Collaborative editing',
 ' Policies and content',
 'Governance',
 'Community',
 'Language editions',
 'Reception',
 'Operation',
 'Access to content',
 'Cultural influence',
 'Related projects',
 'See also',
 'Notes',
 'References',
 'Further reading',
 'External links']

In [ ]:
# Find all paragraphs
p_elements = soup.find_all("p")
ps = [p.get_text() for p in p_elements]
ps

['Wikipedia is a free content online encyclopedia website in 363 languages of the world in which 347 languages are currently active and 16 are closed.[3] It is written and maintained by a  community of volunteers, known as Wikipedians. ',
 'Users can freely use, share and change it, without having to pay. It is also one of the biggest wiki organizations. People can choose to give money to the Wikimedia Foundation to fund Wikipedia and its sister projects. It is an open content website. This means anyone can copy or edit it and make changes to it as long as they follow the rules for copying or editing. ',
 'Wikipedia is owned by an United States organization, the Wikimedia Foundation, which is in San Francisco.',
 "Wikipedia's name is a combination of two words, wiki and encyclopedia.[4]",
 'Wikipedia was started on January 9, 2001, by Jimmy Wales and Larry Sanger as part of an earlier online encyclopedia named Nupedia. On January 15, 2001, Wikipedia became a separate website of its own

In [ ]:
from bs4 import Tag

content = soup.select_one("#mw-content-text .mw-parser-output")
for junk in content.select("span.mw-editsection, .navbox, .navbox-styles, "
                           "style, script, link, meta, .mw-empty-elt"):
    junk.decompose()

def child_tags(section, found):
    # Collect the tag names of a section's content, in order, including subsections.
    for child in section.children:
        if not isinstance(child, Tag):             # skip whitespace
            continue
        if child.name == "section":                # nested h3/h4 subsection → look inside
            child_tags(child, found)
        elif "mw-heading" in (child.get("class") or []):
            h = child.find(["h3", "h4", "h5"])     # record the subheading itself
            if h:
                found.append(h.name)
        else:
            found.append(child.name)
    return found

In [ ]:
# making sense of what else are there in a section of a wiki page
# see what noise are there and whether the section has text for translation

rows = []
for h2 in content.select("div.mw-heading2 > h2"):
    section = h2.find_parent("section")
    tags = child_tags(section, [])
    rows.append({
        "h2": h2.get_text(strip=True),
        "nested_elements": ", ".join(dict.fromkeys(tags)),   # unique, in order of appearance
        "n_p": sum(1 for p in section.find_all("p") if p.get_text(strip=True)),
    })

df_map = pd.DataFrame(rows)
display(df_map)

,h2,nested_elements,n_p
0,History,"figure, div, h3, p",15
1,Collaborative editing,"figure, h3, p, div",11
2,Policies and content,"div, table, p, ul, h3",4
3,Governance,"div, p, h3, h4",7
4,Community,"div, figure, p, h3",6
5,Language editions,"div, p, ul, table, figure, h3",9
6,Reception,"div, p, h3, table, h4",32
7,Operation,"h3, div, figure, p",14
8,Access to content,"div, h3, p, h4, figure",14
9,Cultural influence,"h3, p, h4, div, figure",25
